# NB_Silver_Finance
**PNG Masalais (fictional club, synthetic data)**. Bronze to Silver for finance and registers.

Rule for this layer: *business rules and data quality live here*. The dataflows only connected and shaped.

Principles:
1. **Never drop real money.** Lines with problems are kept and flagged, so totals still match the books.
2. **Only remove what isn't real** (exact duplicate journals from overlapping exports).
3. **Every fix is logged** in `dq_log`, with the original value, so it can be audited.

Default lakehouse: **LH_Bronze** (so `Files/...` paths work). Writes go to **LH_Silver**.
Both lakehouses are created **without** lakehouse schemas, so tables are named `lakehouse.table`.

## 1. Setup and connection test
Checks we can read from bronze and write to silver before doing any real work.
If this cell fails, stop and check both lakehouses are added to the notebook.

In [ ]:
from pyspark.sql import functions as F, Window
from datetime import datetime, timezone
import re

BRONZE = "LH_Bronze"   # lakehouses without schemas: names are lakehouse.table
SILVER = "LH_Silver"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

# Read test
print("gl_au_raw rows:", spark.read.table(f"{BRONZE}.gl_au_raw").count())
# Write test
spark.createDataFrame([(RUN_ID,)], ["run_id"]).write.mode("overwrite").saveAsTable(f"{SILVER}._connection_test")
print("Write to silver OK, run", RUN_ID)

## 2. Helpers
`norm()` turns any column name into lowercase letters and numbers only, so `Debit _AUD_`, `Debit (AUD)`
and `debit aud` all become `debitaud`. We select columns by that normalised key instead of exact names,
which protects us from the dataflow's "Fix it" renames and from small changes in the source system.

`log_issues()` appends rows to the DQ log list, which is written once at the end.

In [ ]:
def norm(name):
    return re.sub(r"[^a-z0-9]", "", name.lower())

def pick(df, mapping):
    """mapping = {normalised_source_key: new_name}. Fails loudly if a column is missing."""
    lookup = {norm(c): c for c in df.columns}
    missing = [k for k in mapping if k not in lookup]
    if missing:
        raise ValueError(f"Columns not found: {missing}. Available: {df.columns}")
    return df.select([F.col(f"`{lookup[k]}`").alias(v) for k, v in mapping.items()])

dq_frames = []
def log_issues(df, source_table, rule, action, key_col, detail_expr):
    dq_frames.append(df.select(
        F.lit(RUN_ID).alias("RunId"),
        F.lit(source_table).alias("SourceTable"),
        F.lit(rule).alias("Rule"),
        F.lit(action).alias("Action"),
        F.col(key_col).cast("string").alias("RecordKey"),
        detail_expr.cast("string").alias("Detail"),
    ))

## 3. Australian GL (Xero-style export)
- Dates arrive as text `dd/MM/yyyy`. We parse with that exact pattern. Any row that fails to parse is logged.
- Debit and Credit become one signed `Amount` (debits positive, credits negative), matching the PNG system.

In [ ]:
au = pick(spark.read.table(f"{BRONZE}.gl_au_raw"), {
    "sourcename": "SourceFile", "date": "DateText", "journal": "JournalRef",
    "accountcode": "AccountCode", "description": "Narration",
    "trackingprogramme": "ProgrammeID", "trackingfund": "AgreementID",
    "debitaud": "Debit", "creditaud": "Credit",
})
au = (au
      .withColumn("TxnDate", F.to_date("DateText", "dd/MM/yyyy"))
      .withColumn("Amount", F.coalesce("Debit", F.lit(0.0)) - F.coalesce("Credit", F.lit(0.0)))
      .withColumn("Entity", F.lit("AU")).withColumn("Currency", F.lit("AUD")))

bad_dates = au.filter(F.col("TxnDate").isNull())
log_issues(bad_dates, "gl_au_raw", "Unparseable date", "Flagged", "JournalRef", F.col("DateText"))
print("AU rows:", au.count(), "| bad dates:", bad_dates.count())

### 3a. Remove duplicate journals from overlapping exports
The March 2026 file repeats February's lines. A journal number is unique in the accounting system,
so a second copy of the same `JournalRef` is not a second transaction. We keep the copy from the
**earliest** file (its home month) and log the rest.

In [ ]:
w = Window.partitionBy("JournalRef").orderBy("SourceFile")
au = au.withColumn("_rn", F.row_number().over(w))
dups = au.filter("_rn > 1")
log_issues(dups, "gl_au_raw", "Duplicate journal (export overlap)", "Removed", "JournalRef",
           F.concat_ws(" | ", "SourceFile", "DateText", F.col("Amount").cast("string")))
au = au.filter("_rn = 1").drop("_rn")
print("Duplicates removed:", dups.count(), "| AU rows now:", au.count())

## 4. PNG GL (signed-amount export)
Already has real dates and a signed amount, so mostly renaming.

In [ ]:
png = pick(spark.read.table(f"{BRONZE}.gl_png_raw"), {
    "sourcename": "SourceFile", "txndate": "TxnDate", "ref": "JournalRef", "glaccount": "AccountCode",
    "narration": "Narration", "costcentre": "ProgrammeID", "fundcode": "AgreementID", "amount": "Amount",
})
png = (png.withColumn("TxnDate", F.to_date("TxnDate"))
          .withColumn("Entity", F.lit("PNG")).withColumn("Currency", F.lit("PGK")))
print("PNG rows:", png.count())

## 5. Combine both ledgers into one shape

In [ ]:
cols = ["Entity", "Currency", "TxnDate", "JournalRef", "AccountCode", "ProgrammeID",
        "AgreementID", "Amount", "Narration", "SourceFile"]
gl = au.select(cols).unionByName(png.select(cols))
gl = gl.withColumn("ProgrammeID", F.nullif(F.trim("ProgrammeID"), F.lit(""))) \
       .withColumn("AgreementID", F.nullif(F.col("AgreementID"), F.lit("")))
print("Combined rows:", gl.count())

## 6. Standardise fund codes
Typed variants like `cssp-est`, `CSSP EST` or `CSSP-EST ` (trailing space) are the same fund.
Fix: trim, uppercase, spaces to hyphens. Log every change with the original value.
Then check every fund code exists in the Agreements register.

In [ ]:
gl = gl.withColumn("AgreementID_clean",
                   F.regexp_replace(F.upper(F.trim("AgreementID")), r"\s+", "-"))
changed = gl.filter(F.col("AgreementID").isNotNull() & (F.col("AgreementID") != F.col("AgreementID_clean")))
log_issues(changed, "gl", "Non-standard fund code", "Corrected", "JournalRef",
           F.concat(F.lit("'"), F.col("AgreementID"), F.lit("' -> "), F.col("AgreementID_clean")))
gl = gl.withColumn("AgreementID", F.col("AgreementID_clean")).drop("AgreementID_clean")

valid_funds = spark.read.table(f"{BRONZE}.agreements").select(F.trim("AgreementID").alias("AgreementID"))
unknown_fund = gl.filter(F.col("AgreementID").isNotNull()).join(valid_funds, "AgreementID", "left_anti")
log_issues(unknown_fund, "gl", "Fund code not in Agreements register", "Flagged", "JournalRef", F.col("AgreementID"))
print("Fund codes corrected:", changed.count(), "| still unknown:", unknown_fund.count())

## 7. Expense lines with no programme, and unknown accounts
- Expense lines (accounts starting with 5) must have a programme. Blanks are kept as `UNALLOCATED`, not dropped:
  the money was really spent and must stay in the totals. The board sees it as a separate bar to chase.
- Revenue lines (4xxx) legitimately have no programme.
- Accounts not in the chart of accounts (the suspense account 5999) are kept and flagged.

In [ ]:
accounts = (spark.read.option("header", True).csv("Files/reference/Accounts.csv")
            .select(F.trim("AccountCode").alias("AccountCode"), "AccountName", "AccountType", "CostCategory"))

unalloc = gl.filter(F.col("AccountCode").startswith("5") & F.col("ProgrammeID").isNull())
log_issues(unalloc, "gl", "Expense line missing cost centre", "Flagged (set UNALLOCATED)", "JournalRef",
           F.concat_ws(" | ", "AccountCode", F.col("Amount").cast("string"), "SourceFile"))
gl = gl.withColumn("ProgrammeID",
                   F.when(F.col("AccountCode").startswith("5") & F.col("ProgrammeID").isNull(), F.lit("UNALLOCATED"))
                    .otherwise(F.col("ProgrammeID")))

unknown_acct = gl.join(accounts.select("AccountCode"), "AccountCode", "left_anti")
log_issues(unknown_acct, "gl", "Account not in chart of accounts", "Flagged", "JournalRef",
           F.concat_ws(" | ", "AccountCode", F.col("Amount").cast("string"), "Narration"))

flag = (F.when(F.col("ProgrammeID") == "UNALLOCATED", "Missing cost centre")
         .when(~F.col("AccountCode").isin([r.AccountCode for r in accounts.collect()]), "Unknown account"))
gl = gl.withColumn("DQ_Flag", flag)
print("Unallocated:", unalloc.count(), "| unknown accounts:", unknown_acct.count())

## 8. Convert to AUD for board reporting
PGK amounts are divided by that month's average AUD to PGK rate. AUD amounts stay as they are.
Each agreement will still be judged in its *own* currency in gold. `AmountAUD` is only for combined board totals.

In [ ]:
fx = (spark.read.option("header", True).csv("Files/fx/FX_AUD_PGK_Monthly.csv")
      .select(F.to_date("Month").alias("MonthStart"), F.col("AvgRate").cast("double").alias("AUDtoPGK")))

gl = (gl.withColumn("MonthStart", F.trunc("TxnDate", "month"))
        .join(fx, "MonthStart", "left")
        .withColumn("AmountAUD", F.when(F.col("Currency") == "AUD", F.col("Amount"))
                                  .otherwise(F.round(F.col("Amount") / F.col("AUDtoPGK"), 2))))

missing_fx = gl.filter(F.col("AmountAUD").isNull())
log_issues(missing_fx, "gl", "No FX rate for month", "Flagged", "JournalRef", F.col("MonthStart"))
print("Rows missing FX:", missing_fx.count())

## 9. Write silver tables

In [ ]:
(gl.select("Entity", "TxnDate", "MonthStart", "JournalRef", "AccountCode", "ProgrammeID", "AgreementID",
           "Currency", F.col("Amount").alias("AmountLocal"), "AUDtoPGK", "AmountAUD", "Narration",
           "SourceFile", "DQ_Flag")
   .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.gl"))

accounts.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.accounts")
fx.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.fx_rates")

budget = pick(spark.read.table(f"{BRONZE}.budget"), {
    "agreementid": "AgreementID", "programmeid": "ProgrammeID", "costcategory": "CostCategory",
    "currency": "Currency", "monthstart": "MonthStart", "budgetamount": "BudgetAmount"})
budget.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.budget")

# Registers: trim every text column, keep the rest as is
for t in ["funders", "agreements", "programmes", "milestones", "milestone_updates"]:
    df = spark.read.table(f"{BRONZE}.{t}")
    df = df.select([F.trim(F.col(c)).alias(c) if dt == "string" else F.col(c) for c, dt in df.dtypes])
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.{t}")
print("Silver tables written")

## 10. Write the DQ log
Appended, not overwritten, so each run's findings are kept with its `RunId`. The report shows the latest run.

In [ ]:
from functools import reduce
dq = reduce(lambda a, b: a.unionByName(b), dq_frames).withColumn("LoggedAtUtc", F.current_timestamp())
dq.write.mode("append").saveAsTable(f"{SILVER}.dq_log")
display(dq.groupBy("SourceTable", "Rule", "Action").count().orderBy("SourceTable", "Rule"))

## 11. Reconciliation checks
Expected (from the generator answer key): GL 3,809 rows (186 AU + 3,623 PNG); 14 duplicates removed;
71 fund codes corrected; 28 unallocated; 6 unknown account. Money check: silver totals must equal bronze
totals minus only the removed duplicates.

In [ ]:
s = spark.read.table(f"{SILVER}.gl")
print("Silver GL rows:", s.count())
display(s.groupBy("Entity", "Currency").agg(F.count("*").alias("Rows"),
                                            F.round(F.sum("AmountLocal"), 2).alias("NetAmount")))

png_bronze = spark.read.table(f"{BRONZE}.gl_png_raw").agg(F.round(F.sum("Amount"), 2)).first()[0]
png_silver = s.filter("Entity = 'PNG'").agg(F.round(F.sum("AmountLocal"), 2)).first()[0]
print("PNG bronze vs silver net:", png_bronze, png_silver, "MATCH" if png_bronze == png_silver else "MISMATCH")
spark.sql(f"DROP TABLE IF EXISTS {SILVER}._connection_test")